# Phần 1: Bài toán Hồi quy
**Dataset:** California Housing Prices (Kaggle / StatLib)  

---

## Mục lục
- [B. EDA & Tiền xử lý Dữ liệu](#b)
  - [B.1 Load & Mô tả dataset](#b1)
  - [B.2 Xử lý Missing Values](#b2)
  - [B.3 Phân bố biến mục tiêu](#b3)
  - [B.4 Ma trận tương quan & Scatter plots](#b4)
  - [B.5 Phát hiện Outliers](#b5)
  - [B.6 Feature Engineering & Encode Categorical](#b6)
  - [B.7 Stratified Train/Val/Test Split](#b7)
  - [B.8 Chuẩn hóa (StandardScaler)](#b8)
  - [B.9 Kiểm tra phân phối sau split](#b9)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

# Import toàn bộ helper functions từ utils.py
from utils import *

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = '../../data/housing.csv' 
print('Import thành công!')

---
<a id='b'></a>
## B. Phân tích Khám phá & Tiền xử lý Dữ liệu

Phần này thực hiện đầy đủ pipeline EDA trước khi huấn luyện mô hình, bao gồm:
phân tích thống kê mô tả, phát hiện và xử lý missing values, phát hiện outliers,
phân tích tương quan, feature engineering, và chia dữ liệu train/val/test.

<a id='b1'></a>
### B.1 Load & Mô tả Dataset

**Dataset:** California Housing Prices, thu thập từ Census 1990 của California.  
Mỗi mẫu đại diện cho một **block group** (đơn vị địa lý nhỏ nhất của census,
thường có 600–3000 người).

| Feature | Kiểu | Mô tả |
|---|---|---|
| `longitude` | float | Kinh độ của block group |
| `latitude` | float | Vĩ độ của block group |
| `housing_median_age` | float | Tuổi trung vị của nhà trong block group |
| `total_rooms` | float | Tổng số phòng trong block group |
| `total_bedrooms` | float | Tổng số phòng ngủ (có NaN) |
| `population` | float | Dân số block group |
| `households` | float | Số hộ gia đình |
| `median_income` | float | Thu nhập trung vị (đơn vị: 10,000 USD) |
| `median_house_value` | float | **Target** — Giá nhà trung vị ($) |
| `ocean_proximity` | object | Khoảng cách tới biển (5 nhóm) |


In [ ]:
df = load_data(DATA_PATH)

In [ ]:
# Thống kê mô tả mở rộng (thêm skewness & kurtosis)
stats_df = describe_stats(df)
stats_df

**Nhận xét:**
- `total_rooms`, `total_bedrooms`, `population`, `households` có **skewness cao** (> 2),
  phân phối lệch phải mạnh — cần lưu ý khi dùng các phương pháp giả thiết Gaussian.
- `housing_median_age` bị **cap ở 52** (max = 52), tương tự `median_house_value` bị cap ở $500,000.
- `median_income` không có đơn vị USD thông thường mà đã được scale (đơn vị ≈ $10,000).

<a id='b2'></a>
### B.2 Xử lý Missing Values

Theo thống kê, chỉ có `total_bedrooms` chứa giá trị NaN.
Ta chọn **impute bằng median** vì:
1. Phân phối của `total_bedrooms` lệch phải mạnh → mean bị kéo bởi outliers.
2. Median robust hơn với outliers trong trường hợp này.

In [ ]:
# Báo cáo missing values
_ = report_missing(df)

In [ ]:
# Impute total_bedrooms bằng median
df = impute_missing(df, col='total_bedrooms', strategy='median')

# Xác nhận không còn NaN
print(f"Missing sau impute: {df.isnull().sum().sum()}")

<a id='b3'></a>
### B.3 Phân bố Biến Mục tiêu (`median_house_value`)

Trước khi xây dựng mô hình, cần hiểu rõ phân bố của biến mục tiêu.
Đây là bước quan trọng để phát hiện các vấn đề như:
- **Ceiling effect**: giá trị bị giới hạn cứng
- **Heavy-tailed distribution**: ảnh hưởng đến assumption của OLS
- **Multimodality**: có thể cần phân tách dữ liệu

In [ ]:
plot_target_distribution(df, target='median_house_value')

**Nhận xét:**
- Phân phối lệch phải, với đuôi dài về phía giá trị cao.
- Có **spike rõ rệt tại \$500,000** do **ceiling effect** — Census đặt cap tại mức này.
  Các mẫu này (~5%) không phản ánh giá trị thực → có thể ảnh hưởng đến mô hình.
- Mean (≈ \$206k) > Median (≈ \$179k), xác nhận phân phối lệch phải.
- **Hướng xử lý:** giữ lại nhưng ghi chú; có thể thử log-transform target ở phần C.

<a id='b4'></a>
### B.4 Ma trận Tương quan & Scatter Plots

Phân tích tương quan giúp:
1. Xác định features có **tương quan cao với target** → ưu tiên trong mô hình.
2. Phát hiện **multicollinearity** giữa các features → ảnh hưởng đến OLS.
3. Định hướng **feature engineering** (tạo ratio features).

In [ ]:
target_corr = plot_correlation_matrix(df, target='median_house_value')

In [ ]:
# Scatter plots các features quan trọng nhất với target
top_features = target_corr.abs().sort_values(ascending=False).head(6).index.tolist()
print(f"Top 6 features tương quan với target: {top_features}")
plot_scatter_features(df, features=top_features, target='median_house_value')

**Nhận xét:**
- `median_income` có tương quan cao nhất với target (r ≈ 0.69) — đây là predictor quan trọng nhất.
- `total_rooms`, `total_bedrooms`, `households`, `population` tương quan cao với nhau
  (multicollinearity) → Ridge/Lasso sẽ xử lý tốt hơn OLS thuần túy.
- `latitude` và `longitude` có tương quan âm — phản ánh hiệu ứng địa lý
  (vùng ven biển phía nam giá cao hơn).

<a id='b5'></a>
### B.5 Phát hiện Outliers

Hai phương pháp được sử dụng:

- **IQR method:** điểm nằm ngoài $[Q_1 - 1.5 \cdot IQR,\ Q_3 + 1.5 \cdot IQR]$  
  Phù hợp hơn cho phân phối lệch, không giả thiết Gaussian.

- **Z-score method:** điểm có $|z| > 3$ (nằm ngoài 3 độ lệch chuẩn)  
  Giả thiết phân phối Gaussian, nhạy hơn với heavy-tailed distributions.

Ta áp dụng trên các biến **tổng** (`total_rooms`, `population`, v.v.) vì chúng phụ thuộc vào
kích thước block group → phân phối lệch mạnh.

In [ ]:
outlier_report = report_outliers(df)
outlier_report

**Nhận xét:**
- `total_rooms`, `total_bedrooms`, `population`, `households` có outliers theo IQR 
  (~5–6%), phản ánh sự chênh lệch lớn về quy mô giữa các block group.
- Z-score cho kết quả ít outliers hơn vì phân phối không Gaussian (không thỏa mãn giả thiết).
- **Quyết định:** Giữ lại outliers vì đây là dữ liệu thực tế hợp lệ, không phải lỗi đo lường.
  Regularization (Ridge/Lasso) sẽ giúp giảm ảnh hưởng của chúng.

<a id='b6'></a>
### B.6 Feature Engineering & Encode Categorical

**Ratio features:** Các biến tổng (`total_rooms`, `population`) phụ thuộc vào kích thước block group,
nên ít có ý nghĩa trực tiếp. Tạo thêm ratio features có ý nghĩa thực tế hơn:

$$\text{rooms\_per\_household} = \frac{\text{total\_rooms}}{\text{households}}$$

$$\text{bedrooms\_per\_room} = \frac{\text{total\_bedrooms}}{\text{total\_rooms}}$$

$$\text{population\_per\_household} = \frac{\text{population}}{\text{households}}$$

**One-hot encoding:** `ocean_proximity` là biến categorical với 5 giá trị không có thứ tự
→ cần encode thành các biến dummy (không dùng `drop_first` để giữ khả năng giải thích).

In [ ]:
# Tạo ratio features
df = engineer_features(df)

In [ ]:
# Kiểm tra tương quan của features mới với target
new_features = ['rooms_per_household', 'bedrooms_per_room', 'population_per_household']
for f in new_features:
    from scipy.stats import pearsonr
    r, p = pearsonr(df[f], df['median_house_value'])
    print(f"  {f:<35} r = {r:+.4f}  (p = {p:.2e})")

In [ ]:
# One-hot encode ocean_proximity
df = encode_categorical(df, col='ocean_proximity')
print(f"\nShape sau encoding: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

**Nhận xét:**
- `rooms_per_household` có tương quan **dương** với target (nhà nhiều phòng hơn thường đắt hơn).
- `bedrooms_per_room` có tương quan **âm** (tỷ lệ phòng ngủ cao → nhà chất lượng thấp hơn).
- `population_per_household` có tương quan **âm** (hộ đông người → khu vực thu nhập thấp hơn).
- Các ratio features này sẽ được dùng làm **basis tự chọn** trong phần E.

<a id='b7'></a>
### B.7 Stratified Train / Val / Test Split

**Tại sao stratified?**  
`median_income` là predictor quan trọng nhất và có phân phối không đều.
Nếu chia ngẫu nhiên, một tập có thể thiếu representation của nhóm thu nhập cao/thấp.
Ta chia `median_income` thành 5 nhóm và dùng `StratifiedShuffleSplit` để đảm bảo
tỷ lệ mỗi nhóm **nhất quán** trong cả 3 tập.

**Tỷ lệ:** 70% train / 10% val / 20% test

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, feature_names = stratified_split(
    df,
    target='median_house_value',
    test_size=0.2,
    val_size=0.1,
    random_state=RANDOM_STATE
)

print(f"\nFeatures ({len(feature_names)}):")
for i, f in enumerate(feature_names):
    print(f"  [{i:>2}] {f}")

<a id='b8'></a>
### B.8 Chuẩn hóa Features (StandardScaler)

**Tại sao cần chuẩn hóa?**
- Các features có **đơn vị khác nhau**: `total_rooms` (hàng nghìn) vs `latitude` (32–38).
- OLS với Normal Equations không yêu cầu scale, nhưng **Gradient Descent** và
  **Regularization (Ridge/Lasso)** rất nhạy với scale của features.
- Ridge và Lasso phạt theo $\|\mathbf{w}\|^2$ hoặc $\|\mathbf{w}\|_1$:
  nếu không scale, features có giá trị lớn sẽ bị phạt nặng hơn một cách không công bằng.

**Quy tắc quan trọng:** Scaler chỉ được **fit trên train set**,
sau đó **transform** val và test. Không được fit trên val/test để tránh **data leakage**.

In [ ]:
X_train_s, X_val_s, X_test_s, scaler = scale_features(X_train, X_val, X_test)

print(f"\nSau scaling:")
print(f"  X_train : {X_train_s.shape}")
print(f"  X_val   : {X_val_s.shape}")
print(f"  X_test  : {X_test_s.shape}")

<a id='b9'></a>
### B.9 Kiểm tra Phân phối sau Split

Xác nhận rằng stratification đã đảm bảo phân phối target nhất quán
giữa 3 tập dữ liệu (không bị bias do quá trình chia).

In [ ]:
plot_split_distribution(y_train, y_val, y_test, target='median_house_value')

In [ ]:
# So sánh thống kê cơ bản giữa 3 tập
summary = pd.DataFrame({
    'Train': pd.Series(y_train).describe(),
    'Val'  : pd.Series(y_val).describe(),
    'Test' : pd.Series(y_test).describe(),
}).round(2)
summary

**Nhận xét:** Mean, median, std của 3 tập xấp xỉ nhau → stratification thành công.

---

## Tổng kết Phần B

| Bước | Kết quả |
|---|---|
| Missing values | 207 NaN trong `total_bedrooms` → impute bằng median |
| Outliers | Giữ lại — dữ liệu thực tế hợp lệ |
| Feature engineering | +3 ratio features, one-hot encode `ocean_proximity` (5→5 dummy cols) |
| Tổng features | 16 features sau encoding |
| Split | 14,448 train / 2,064 val / 4,128 test (stratified) |
| Scaling | StandardScaler fit trên train only |

**Dữ liệu sẵn sàng cho Phần C (Linear Regression):**
- `X_train_s`, `X_val_s`, `X_test_s` — đã scale
- `X_train`, `X_val`, `X_test` — chưa scale (dùng cho Normal Equations)
- `y_train`, `y_val`, `y_test`
- `scaler`, `feature_names`

<a id='c'></a>
## C. Xây dựng và Huấn luyện mô hình

Trong phần này, chúng ta sẽ thực hiện giải bài toán Hồi quy tuyến tính bằng hai cách tiếp cận khác nhau để so sánh hiệu quả và tốc độ hội tụ:
1. **Phương trình chuẩn (Normal Equation):** Giải trực tiếp bằng đại số tuyến tính.
2. **Mini-batch Gradient Descent:** Tối ưu hóa bằng giải thuật giảm đạo hàm.

---

### C.1 Phương trình chuẩn (Normal Equation)

Phương trình chuẩn giúp tìm trực tiếp bộ tham số $\theta$ tối ưu mà không cần thực hiện các vòng lặp:
$$\hat{\theta} = (X^T X)^{-1} X^T y$$

**Lưu ý:** Với phương pháp này, chúng ta sử dụng tập dữ liệu gốc (`X_train`, `X_val`) và thêm một cột bias (giá trị bằng 1).

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

X_train_b = np.c_[np.ones((len(X_train), 1)), X_train].astype(float)
X_val_b = np.c_[np.ones((len(X_val), 1)), X_val].astype(float)

# Tính toán Normal Equation
X_transpose = X_train_b.T
theta_best = np.linalg.inv(X_transpose.dot(X_train_b)).dot(X_transpose).dot(y_train)

# Dự đoán trên tập Validation
y_val_predict = X_val_b.dot(theta_best)

# Tính toán sai số RMSE
rmse_normal = np.sqrt(mean_squared_error(y_val, y_val_predict))

print("--- Kết quả sau khi sửa lỗi ---")
print(f"Số lượng tham số (theta): {len(theta_best)}")
print(f"RMSE trên tập Validation: {rmse_normal:,.2f}")

**Nhận xét về Normal Equation:**
- **Ưu điểm:** Giải trực tiếp, không cần chọn Learning Rate, kết quả là tối ưu tuyệt đối trên tập huấn luyện.
- **Nhược điểm:** Chi phí tính toán ma trận nghịch đảo $O(n^3)$ sẽ rất lớn nếu số lượng đặc trưng (features) tăng cao.
- **Kết quả RMSE:** `857,529.51`

### C.2 Mini-batch Gradient Descent với Learning Rate Schedule

Khác với Normal Equation, Gradient Descent là một giải thuật tối ưu hóa lặp. Chúng ta sử dụng **Mini-batch** để cân bằng giữa tốc độ và sự ổn định, kết hợp với **Cosine Annealing** để điều chỉnh tốc độ học (learning rate) giảm dần theo thời gian, giúp mô hình hội tụ tốt hơn.

**Lưu ý:** Bắt buộc sử dụng dữ liệu đã chuẩn hóa (`X_train_s`, `X_val_s`) để thuật toán hội tụ.

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error

# 1. Thêm cột bias vào dữ liệu đã scale
X_train_sm = np.c_[np.ones((len(X_train_s), 1)), X_train_s].astype(float)
X_val_sm = np.c_[np.ones((len(X_val_s), 1)), X_val_s].astype(float)

# 2. Cấu hình Hyperparameters
n_epochs = 100
batch_size = 32
eta_max = 0.0001
eta_min = 0.000001

def cosine_annealing(epoch, n_epochs, eta_max, eta_min):
    return eta_min + 0.5 * (eta_max - eta_min) * (1 + np.cos(np.pi * epoch / n_epochs))

# 3. Huấn luyện Mini-batch GD
m = len(X_train_sm)
theta_gd = np.random.randn(X_train_sm.shape[1], 1) # Khởi tạo ngẫu nhiên

y_train_rect = np.array(y_train).reshape(-1, 1)
y_val_rect = np.array(y_val).reshape(-1, 1) 
# -----------------------

loss_history = []
start_time = time.time()

for epoch in range(n_epochs):
    shuffled_indices = np.random.permutation(m)
    X_shuffled = X_train_sm[shuffled_indices]
    y_shuffled = y_train_rect[shuffled_indices]
    
    eta = cosine_annealing(epoch, n_epochs, eta_max, eta_min)
    
    for i in range(0, m, batch_size):
        xi = X_shuffled[i:i+batch_size]
        yi = y_shuffled[i:i+batch_size]
        gradients = 2/len(xi) * xi.T.dot(xi.dot(theta_gd) - yi)
        theta_gd = theta_gd - eta * gradients
    
    y_train_pred = X_train_sm.dot(theta_gd)
    loss = np.sqrt(mean_squared_error(y_train_rect, y_train_pred))
    loss_history.append(loss)

print(f"Max theta: {np.max(np.abs(theta_gd))}")

gd_time = time.time() - start_time
print(f"Thời gian huấn luyện GD: {gd_time:.4f}s")

# 4. Dự đoán và so sánh
y_val_pred_gd = X_val_sm.dot(theta_gd)
rmse_gd = np.sqrt(mean_squared_error(y_val_rect, y_val_pred_gd))
print(f"RMSE (GD) trên tập Validation: {rmse_gd:,.2f}")

# Vẽ đồ thị hội tụ
plt.figure(figsize=(8, 5))
plt.plot(loss_history, color='blue', label='Train RMSE')
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("Sự hội tụ của Mini-batch Gradient Descent (Cosine Annealing)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import scipy.stats as stats
import statsmodels.stats.api as sms
from statsmodels.compat import lzip
import matplotlib.pyplot as plt

residuals = np.array(y_val).reshape(-1, 1) - y_val_pred_gd

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# 1. Residual Plot
ax[0].scatter(y_val_pred_gd, residuals, alpha=0.5, color='teal')
ax[0].axhline(y=0, color='r', linestyle='--')
ax[0].set_xlabel("Giá trị dự báo (Predicted)")
ax[0].set_ylabel("Phần dư (Residuals)")
ax[0].set_title("1. Residual Plot")

# 2. QQ-Plot
stats.probplot(residuals.flatten(), dist="norm", plot=ax[1])
ax[1].set_title("2. Normal QQ-plot")
plt.tight_layout()
plt.show()

# 3. Breusch–Pagan Test
test_results = sms.het_breuschpagan(residuals, X_val_sm)
names = ['Lagrange multiplier statistic', 'p-value', 'f-value', 'f p-value']

print("--- KẾT QUẢ KIỂM ĐỊNH BREUSCH–PAGAN ---")
for name, value in zip(names, test_results):
    if 'p-value' in name:
        print(f"{name:<30}: {value:.4e}")
    else:
        print(f"{name:<30}: {value:.4f}")

p_value = test_results[1]
if p_value < 0.05:
    print(f"\nKết luận: p-value ({p_value:.4e}) < 0.05")
    print("=> Bác bỏ H0. Có hiện tượng Heteroscedasticity (Phương sai thay đổi).")
else:
    print(f"\nKết luận: p-value ({p_value:.4e}) > 0.05")
    print("=> Chưa có bằng chứng vi phạm giả thiết phương sai không đổi.")

### C.4 Xử lý Heteroscedasticity bằng Weighted Least Squares (WLS)

Nếu p-value của test Breusch–Pagan < 0.05, ta bác bỏ giả thiết phương sai không đổi. Khi đó, WLS sẽ gán trọng số thấp cho các điểm có phương sai lớn để cải thiện mô hình.

In [ ]:
if test_results[1] < 0.05:
    print("Phát hiện Heteroscedasticity! Đang triển khai WLS...")
    
    # Ước lượng trọng số 
    weights = 1.0 / (np.abs(residuals).flatten() + 1e-5)
    
    import statsmodels.api as sm
    
    wls_model = sm.WLS(y_val, X_val_sm, weights=weights)
    results = wls_model.fit()
    
    print(results.summary())
    y_val_pred_wls = results.predict(X_val_sm)
    rmse_wls = np.sqrt(mean_squared_error(y_val, y_val_pred_wls))
    print(f"\nRMSE sau khi dùng WLS: {rmse_wls:,.2f}")
else:
    print("Không phát hiện Heteroscedasticity nghiêm trọng. Có thể giữ nguyên mô hình OLS.")
